In [1]:
!pip install transformers accelerate mlflow torch scikit-learn evaluate 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 49.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/

In [11]:

from transformers import AutoTokenizer ,AutoModelForSequenceClassification
import mlflow
from transformers.integrations import MLflowCallback
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import load_dataset, DatasetDict 
from sklearn.model_selection import train_test_split
import pandas as pd


In [12]:
MODEL_NAME = "Shushant/nepaliBERT"
NUM_LABELS = 2 # 0 and 1 sentiment


In [13]:
tokenizer = AutoTokenizer.from_pretrained("Shushant/nepaliBERT")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Shushant/nepaliBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
df = pd.read_parquet("/kaggle/input/nepali-sentiment-dataset/nepali_sentiment_dataset.parquet")

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111673 entries, 0 to 111672
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   sentences  111673 non-null  object
 1   sentiment  111673 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.7+ MB


In [16]:
# Check the frequency of each sentiment label
print(df['sentiment'].value_counts())


sentiment
1    57226
0    54447
Name: count, dtype: int64


In [21]:

import os 
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df["sentiment"]
)

train_file = "temp_train.parquet"
val_file = "temp_val.parquet"
train_df.to_parquet(train_file, index=False)
val_df.to_parquet(val_file, index=False)

# 2. Load files into a HF DatasetDict object
raw_datasets = load_dataset("parquet", data_files={"train": train_file, "validation": val_file})


def tokenize_fn(batch):
    
    return tokenizer(batch["sentences"], truncation=True, padding="max_length", max_length=64)

tokenized = raw_datasets.map(tokenize_fn, batched=True)


columns_to_keep = ["input_ids", "attention_mask", "sentiment"]
tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in columns_to_keep])


tokenized = tokenized.rename_column("sentiment", "label") 
tokenized.set_format(type="torch")


train_dataset = tokenized["train"]
eval_dataset = tokenized["validation"]

# --- Cleanup ---
os.remove(train_file)
os.remove(val_file)



Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/89338 [00:00<?, ? examples/s]

Map:   0%|          | 0/22335 [00:00<?, ? examples/s]

In [22]:
from transformers import TrainingArguments, Trainer

In [23]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    # Ensure average="binary" is appropriate for your 0/1 sentiments
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary") 
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

In [25]:
# ...existing code...
training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    fp16=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=["mlflow"],   # enable reporting to MLflow
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[MLflowCallback()]
)

# Optionally create an explicit MLflow run to attach extra tags/params
with mlflow.start_run() as run:
    mlflow.log_param("model_name", "Shushant/nepaliBERT")
    mlflow.log_param("num_labels", 2)
    trainer.train()
    metrics = trainer.evaluate()
    mlflow.log_metrics(metrics)
    trainer.save_model("kaggle/working/outputs/best_model")
# ...existing code...

/tmp/ipykernel_48/2893244381.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You are adding a <class 'transformers.integrations.integration_utils.MLflowCallback'> to the callbacks of this Trainer, but there is already one. The currentlist of callbacks is
:DefaultFlowCallback
MLflowCallback
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.356300,0.315629,0.863667,0.895778,0.830581,0.861949
2,0.252400,0.297181,0.885516,0.892857,0.882481,0.887639
3,0.188100,0.297657,0.892590,0.895782,0.894452,0.895117


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [4]:
!zip -r all_working_files.zip .

  adding: mlruns/ (stored 0%)
  adding: mlruns/.trash/ (stored 0%)
  adding: mlruns/0/ (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/ (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/ (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/save_only_model (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/deepspeed (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/include_num_input_tokens_seen (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/finetuning_task (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/liger_kernel_config (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/add_cross_attention (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/fp16_full_eval (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/params/torchdynamo (stored 0%)
  adding: mlruns/0/a952ed589ba947beb89eaed8d6ca3fcf/param

In [5]:
!ls model_outputs.zip


ls: cannot access 'model_outputs.zip': No such file or directory


In [6]:
from IPython.display import FileLink

# Generates a link to download the file created above
FileLink(r'all_working_files.zip')

/kaggle/working/all_working_files.zip

In [7]:
!rm all_working_files.zip

In [ ]:
cd